In [32]:
import pandas as pd
import re
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from transformers import DistilBertTokenizerFast
from sklearn.preprocessing import OneHotEncoder
import torch
from deep_translator import GoogleTranslator

In [31]:
def translate_to_english(text):
    try:
        return GoogleTranslator(source='auto', target='en').translate(text)
    except:
        return text  # fallback if translation fails


### Load Dataset

In [33]:

file = "../datasets/emotion-recognition-dataset/data/train-00000-of-00001.parquet"
df = pd.read_parquet(file)
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

In [ ]:
df['Text'] = df['Text'].apply(translate_to_english)

### Clean dataset

In [35]:
def basic_clean(text):
    text = re.sub(r'http\S+', '', text)  # Remove URLs
    text = re.sub(r'@\w+', '', text)     # Remove mentions
    text = re.sub(r'#\w+', '', text)     # Remove hashtags
    text = re.sub(r'\s+', ' ', text)     # Remove extra whitespace
    text = re.sub(r"<.*?>", "", text)    # Remove HTML tags
    text = text.strip()                   # Remove leading/trailing whitespace
    return text

df['clean_text'] = df['Text'].apply(basic_clean)

In [36]:
df

,Text,Emotion,clean_text
0,Why ?,neutral,Why ?
1,Sage Act upgrade on my to do list for tommorow.,joy,Sage Act upgrade on my to do list for tommorow.
2,ON THE WAY TO MY HOMEGIRL BABY FUNERAL!!! MAN ...,sadness,ON THE WAY TO MY HOMEGIRL BABY FUNERAL!!! MAN ...
3,Such an eye ! The true hazel eye-and so brill...,joy,Such an eye ! The true hazel eye-and so brilli...
4,@Iluvmiasantos ugh babe.. hugggzzz for u .! b...,joy,ugh babe.. hugggzzz for u .! babe naamazed nga...
...,...,...,...
595357,I just found out I have a twin I never knew ab...,surprise,I just found out I have a twin I never knew ab...
595358,This new technology is unbelievable!,surprise,This new technology is unbelievable!
595359,My favorite celebrity just replied to my tweet!,surprise,My favorite celebrity just replied to my tweet!
595360,I never thought I'd see this in my lifetime!,surprise,I never thought I'd see this in my lifetime!


In [28]:
le = OneHotEncoder(sparse_output=False)
encoded = le.fit_transform(df[['Emotion']])

le_df = pd.DataFrame(encoded, columns=le.get_feature_names_out(['Emotion']))
df = pd.concat([df, le_df], axis=1)

In [30]:
print("Emotion -> One Hot Encoded Mapping:")
for emotion, idx in zip(le.categories_[0], range(len(le.categories_[0]))):
    print(f"{emotion} = {le_df.iloc[idx].values}")

df

Emotion -> One Hot Encoded Mapping:
anger = [0. 0. 0. 0. 1. 0. 0.]
fear = [0. 0. 1. 0. 0. 0. 0.]
joy = [0. 0. 0. 0. 0. 1. 0.]
love = [0. 0. 1. 0. 0. 0. 0.]
neutral = [0. 0. 1. 0. 0. 0. 0.]
sadness = [0. 1. 0. 0. 0. 0. 0.]
surprise = [0. 0. 0. 0. 0. 1. 0.]


,Text,Emotion,clean_text,Emotion_anger,Emotion_fear,Emotion_joy,Emotion_love,Emotion_neutral,Emotion_sadness,Emotion_surprise
0,Why ?,neutral,Why ?,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,Sage Act upgrade on my to do list for tommorow.,joy,Sage Act upgrade on my to do list for tommorow.,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,ON THE WAY TO MY HOMEGIRL BABY FUNERAL!!! MAN ...,sadness,ON THE WAY TO MY HOMEGIRL BABY FUNERAL!!! MAN ...,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,Such an eye ! The true hazel eye-and so brill...,joy,Such an eye ! The true hazel eye-and so brilli...,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,@Iluvmiasantos ugh babe.. hugggzzz for u .! b...,joy,ugh babe.. hugggzzz for u .! babe naamazed nga...,0.0,0.0,1.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
595357,I just found out I have a twin I never knew ab...,surprise,I just found out I have a twin I never knew ab...,0.0,0.0,0.0,0.0,0.0,0.0,1.0
595358,This new technology is unbelievable!,surprise,This new technology is unbelievable!,0.0,0.0,0.0,0.0,0.0,0.0,1.0
595359,My favorite celebrity just replied to my tweet!,surprise,My favorite celebrity just replied to my tweet!,0.0,0.0,0.0,0.0,0.0,0.0,1.0
595360,I never thought I'd see this in my lifetime!,surprise,I never thought I'd see this in my lifetime!,0.0,0.0,0.0,0.0,0.0,0.0,1.0


### Train Test Split

In [ ]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['clean_text'].tolist(),
    df['label_encoded'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df['label_encoded']
)

### DistilBERT Tokenization

In [14]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=128,
)

test_encodings = tokenizer(
    test_texts,
    truncation=True,
    padding=True,
    max_length=128,
)


D:\dev\Python Projects\MindEase\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\avish\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


NameError: name 'train_texts' is not defined